# Assignment 10 — Healthcare Cognitive Assistant (Diabetes Care)

**Parth Mahajan | Roll No. 1024170295 | Batch 3Q31 / 3Q3A**


In [ ]:
import nltk
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(resource, quiet=True)


## Q1

The odd roll number selects the Diabetes Care Information Assistant. It provides general educational information and escalates emergency symptoms to local emergency services. It does not diagnose or replace clinical advice.


In [ ]:
import re, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

examples = {
"glucose_monitoring": ["How often should I check my blood sugar?", "What does a fasting glucose reading mean?", "How do I log my glucose readings?", "When should I test after eating?", "What is a continuous glucose monitor?", "How can I understand my glucose trends?", "Should I record readings before meals?", "What does my meter measure?"],
"medication": ["How should I store insulin?", "What is the general purpose of metformin?", "I forgot whether I took my medicine, what should I do?", "Can you explain insulin types?", "Where should insulin pens be kept?", "What questions should I ask about side effects?", "How do I keep a medication list?", "Why is medication timing important?"],
"nutrition": ["What are balanced meal ideas for diabetes?", "How do carbohydrates affect blood sugar?", "Can you suggest a lower sugar snack?", "What is a serving size?", "How can I read nutrition labels?", "What are sources of dietary fiber?", "How can I plan regular meals?", "What drinks have less added sugar?"],
"activity": ["What kind of physical activity can support health?", "How can I start walking safely?", "Why is movement helpful for glucose management?", "How do I plan activity during the week?", "What should I discuss before starting exercise?", "Can short walks count as activity?", "How can I track daily movement?", "What are gentle exercise options?"],
"appointments": ["What should I ask my diabetes care team?", "How often should I schedule checkups?", "What information should I bring to an appointment?", "How do I prepare for an A1C discussion?", "What is an A1C test generally used for?", "How can I find a diabetes educator?", "What should I mention about recent readings?", "How can I organize questions for my clinician?"]
}
records=[(query,intent) for intent,queries in examples.items() for query in queries]
data=pd.DataFrame(records,columns=["query","intent"])
print("Labeled queries:",len(data),"categories:",data.intent.nunique())
display(data.groupby("intent").size().rename("count").to_frame())

## Q2


In [ ]:
taxonomy = {
    "monitoring": ["glucose_readings", "monitoring_devices"],
    "treatment": ["medications", "medicine_safety"],
    "daily_living": ["nutrition", "physical_activity"],
    "professional_care": ["appointments", "care_team_support"],
}
taxonomy

## Q3


In [ ]:
concepts = {
    "Person": "individual managing diabetes", "BloodGlucose": "measured glucose value", "GlucoseMeter": "device measuring a glucose sample", "CGM": "device tracking glucose over time", "Insulin": "prescribed glucose-lowering medicine", "Medication": "medicine in an individual care plan", "Meal": "food consumed", "Carbohydrate": "nutrient affecting glucose", "Activity": "physical movement", "A1C": "laboratory measure reflecting longer-term glucose", "Clinician": "qualified healthcare professional", "Appointment": "scheduled clinical visit"
}
relationships = [
    ("Person", "records", "BloodGlucose"), ("GlucoseMeter", "measures", "BloodGlucose"), ("CGM", "estimates", "BloodGlucose"),
    ("Person", "may be prescribed", "Medication"), ("Insulin", "is a type of", "Medication"), ("Meal", "contains", "Carbohydrate"),
    ("Carbohydrate", "can influence", "BloodGlucose"), ("Activity", "can influence", "BloodGlucose"),
    ("Clinician", "reviews", "A1C"), ("Appointment", "connects Person with", "Clinician"),
    ("Person", "attends", "Appointment"), ("Person", "uses", "GlucoseMeter")
]
print("Concepts:",len(concepts),"Relationships:",len(relationships))
display(pd.DataFrame(relationships,columns=["subject","relation","object"]))

## Q4


In [ ]:
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
stop_words=set(stopwords.words("english")); lemmatizer=WordNetLemmatizer()
def preprocess(text):
    tokens=re.findall(r"[a-z]+",text.lower())
    return " ".join(lemmatizer.lemmatize(token) for token in tokens if token not in stop_words)
X_train,X_test,y_train,y_test=train_test_split(data.query,data.intent,test_size=0.25,random_state=42,stratify=data.intent)
model=Pipeline([("tfidf",TfidfVectorizer(preprocessor=preprocess,ngram_range=(1,2))), ("classifier",LogisticRegression(max_iter=1000,class_weight="balanced"))])
model.fit(X_train,y_train); predictions=model.predict(X_test)
print("Accuracy:",accuracy_score(y_test,predictions))
precision,recall,f1,_=precision_recall_fscore_support(y_test,predictions,average="macro",zero_division=0)
print("Macro precision:",precision,"Macro recall:",recall,"Macro F1:",f1)
print(classification_report(y_test,predictions,zero_division=0))
display(pd.DataFrame(confusion_matrix(y_test,predictions,labels=model.classes_),index=model.classes_,columns=model.classes_))

The preprocessing lowercases, tokenizes with a regular expression, removes English stopwords, and lemmatizes. TF-IDF features feed a Logistic Regression classifier. With a small illustrative corpus, held-out scores are only a demonstration and should not be treated as clinical performance.

## Q5


In [ ]:
knowledge_base = {
    "glucose_monitoring": "A glucose meter or continuous glucose monitor can help track readings. Follow the instructions from your care team and device manufacturer; discuss patterns with a clinician.",
    "medication": "Use medicines only as prescribed. Ask a pharmacist or clinician about missed doses, storage, interactions, and side effects; do not change a dose based on this assistant.",
    "nutrition": "A balanced eating plan is individual. A registered dietitian or diabetes care professional can help tailor meals and explain carbohydrate and nutrition labels.",
    "activity": "Regular physical activity can support general health. Choose an activity appropriate for your abilities and ask your care team what is safe for your situation.",
    "appointments": "Bring medication lists, questions, and recent glucose records to appointments. Your clinician can interpret results such as A1C in your personal context."
}
emergency_pattern=re.compile(r"\\b(chest pain|trouble breathing|unconscious|seizure|severe confusion|cannot wake|fainting)\\b",re.I)
medical_terms={"diabetes","glucose","blood sugar","insulin","medicine","medication","a1c","carbohydrate","exercise","nutrition","health","symptom"}
def respond(query, threshold=0.32):
    if emergency_pattern.search(query):
        return "This may be an emergency. Contact your local emergency number now or go to the nearest emergency department. Do not wait for an online response."
    if not any(term in query.lower() for term in medical_terms):
        return "I can help with general diabetes education. This question appears outside my supported topics; please ask a clinician or a relevant service."
    probabilities=model.predict_proba([query])[0]
    best_index=probabilities.argmax(); confidence=float(probabilities[best_index]); intent=model.classes_[best_index]
    if confidence < threshold:
        return "I am not confident which diabetes topic you mean. Could you rephrase? For personal advice, contact your care team."
    return knowledge_base.get(intent,"I do not have supported information for that topic. Please consult your care team.")

test_queries=[
    "How can I track glucose readings?", "Who can explain my medicine?", "What are balanced meal ideas?",
    "How do I start walking?", "What should I bring to my appointment?", "What should I do with sugar levels?",
    "How do I fix my car engine?", "Can you interpret this vague thing for me?", "I have chest pain and trouble breathing"
]
for query in test_queries: print("Q:",query,"\nA:",respond(query),"\n")

## Q6


In [ ]:
def interactive_assistant():
    print("Diabetes Care Information Assistant. Type 'quit' to finish.")
    while True:
        query=input("You: ").strip()
        if query.lower() in {"quit","exit"}: break
        print("Assistant:",respond(query))

interactive_assistant()

### Answer

1. Emergency symptom phrases trigger immediate emergency escalation.
2. Queries outside supported diabetes topics receive an unsupported-query response.
3. Low-confidence predictions request clarification.
4. Supported intents return topic-specific knowledge-base content.
5. Medication answers direct personalized dosing questions to a pharmacist or clinician.
6. The assistant does not diagnose or change treatment.